# Módulo de análise geo espacial para identificação de desertos médicos Sus

**Autora:** Vanessa Batista ([@vandromedae](https://github.com/vandromedae))  
**Repositório:** [github.com/vandromedae/desertos-medicos-sus](https://github.com/vandromedae/desertos-medicos-sus)  
**Licença:** [MIT](https://github.com/vandromedae/desertos-medicos-sus/blob/main/LICENSE)

---

In [ ]:
# ============================================================
# Desertos Médicos SUS - Análise Geoespacial
# ============================================================
# Objetivo: Cruzar médicos com setores censitários via spatial join
#           e calcular densidade médica em nível de setor.

import sys
from pathlib import Path

# Adicionar raiz do projeto ao path
notebook_path = Path().resolve()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Imports (agora sabemos que são rápidos!)
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_PROCESSED, DATA_EXTERNAL, OUTPUT_MAPAS,
    COMPETENCIA_DEFAULT, IBGE_SP_CAPITAL,
    LIMIAR_DENSIDADE_MEDICA,
)
from src.analysis import classificar_densidade_medica
from src.analysis.e2sfca import (
    peso_gaussiano,
    calcular_percentis_e2sfca,
    classificar_e2sfca,
    calcular_e2sfca,
)
from src.validation.e2sfca import (
    sao_iguais,
    erro_relativo,
    calcular_e2sfca_manual,
    validar_e2sfca_100_setores,
)
from src.geospatial import (
    criar_geodataframe_pontos,
    associar_pontos_a_setores,
)

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print(" Ambiente configurado!")
print(f" Raiz do projeto: {project_root}")
print(f" Dados processados: {DATA_PROCESSED}")

In [ ]:
# ============================================================
# Carregar dados geoespaciais (shapefile com população)
# ============================================================

print("=" * 70)
print("CARREGANDO DADOS GEOESPACIAIS")
print("=" * 70)

# 1. Médicos com coordenadas
arquivo_medicos_geo = DATA_PROCESSED / "medicos_sus_com_coordenadas.parquet"

if not arquivo_medicos_geo.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo_medicos_geo}\n"
        "Execute o notebook 02_limpeza_e_transformacao.ipynb primeiro."
    )

df_medicos = pd.read_parquet(arquivo_medicos_geo)
print(f" Médicos com coordenadas: {len(df_medicos):,} registros")

# 2. Coluna v0001 = população total do setor censitário (IBGE)
shapefile_path = DATA_EXTERNAL / "SP_setores_CD2022_IBGE" / "SP_setores_CD2022.shp"

if not shapefile_path.exists():
    raise FileNotFoundError(
        f"Shapefile não encontrado: {shapefile_path}\n"
        "Baixe de: https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/shp/UF/SP/"
    )

print(f"\n Carregando shapefile dos setores censitários...")
gdf_setores = gpd.read_file(shapefile_path)
print(f" Shapefile carregado: {len(gdf_setores):,} setores")
print(f"   CRS: {gdf_setores.crs}")

# 3. Verificar se a coluna de população está presente
if 'v0001' not in gdf_setores.columns:
    raise ValueError(
        "Coluna 'v0001' (população) não encontrada no shapefile!\n"
        f"Colunas disponíveis: {gdf_setores.columns.tolist()[:10]}"
    )

# 4. Diagnóstico de qualidade da população
n_nulos = gdf_setores['v0001'].isna().sum()
print(f"\n Qualidade dos dados de população:")
print(f"   População total: {gdf_setores['v0001'].fillna(0).sum():,.0f}")
print(f"   Setores sem população: {n_nulos:,} ({n_nulos/len(gdf_setores)*100:.2f}%)")

if n_nulos > 0:
    print(f"\n Diagnóstico dos {n_nulos:,} setores sem população:")
    sem_pop = gdf_setores[gdf_setores['v0001'].isna()]
    if 'SITUACAO' in sem_pop.columns:
        print(f"   Por situação: {sem_pop['SITUACAO'].value_counts().to_dict()}")
    if 'NM_MUN' in sem_pop.columns:
        print(f"   Top municípios: {sem_pop['NM_MUN'].value_counts().head(5).to_dict()}")

# 5. Criar código de município de 6 dígitos (IBGE) se não existir
if 'cod_mun_ibge' not in gdf_setores.columns:
    gdf_setores['cod_mun_ibge'] = gdf_setores['CD_MUN'].astype(str).str[:6]
    print(f"\n Criada coluna 'cod_mun_ibge' a partir de CD_MUN")

# 6. Base municipal (para validação)
arquivo_municipal = DATA_PROCESSED / "base_municipal_densidade_medica.parquet"
if arquivo_municipal.exists():
    df_municipal = pd.read_parquet(arquivo_municipal)
    print(f"\n Base municipal: {len(df_municipal):,} municípios")
else:
    df_municipal = None
    print("\n Base municipal não encontrada (opcional)")

print(f"\n Resumo:")
print(f"   Médicos únicos: {df_medicos['profissional_cns'].nunique():,}")
print(f"   CNES únicos: {df_medicos['cnes'].nunique():,}")
print(f"   Setores censitários: {len(gdf_setores):,}")
print(f"   População total: {gdf_setores['v0001'].fillna(0).sum():,.0f}")

In [ ]:
# ============================================================
# Preparar GeoDataFrames
# ============================================================

print("=" * 70)
print("PREPARANDO GEODATAFRAMES")
print("=" * 70)

# 1. GeoDataFrame de médicos (pontos)
print("\n1. Criando GeoDataFrame de médicos...")
gdf_medicos = gpd.GeoDataFrame(
    df_medicos,
    geometry=gpd.points_from_xy(df_medicos['longitude'], df_medicos['latitude']),
    crs='EPSG:4326'
)
print(f"    {len(gdf_medicos):,} pontos de médicos criados")
print(f"    CRS: {gdf_medicos.crs}")

# 2. Shapefile dos setores (já é GeoDataFrame!)
print("\n2. Shapefile dos setores censitários (com população):")
print(f"    {len(gdf_setores):,} polígonos carregados")
print(f"    CRS: {gdf_setores.crs}")
print(f"    População total: {gdf_setores['v0001'].sum():,}")

# 3. Garantir mesmo CRS
if gdf_medicos.crs != gdf_setores.crs:
    print("\n3. Convertendo médicos para CRS dos setores...")
    gdf_medicos = gdf_medicos.to_crs(gdf_setores.crs)
    print(f"    CRS unificado: {gdf_medicos.crs}")
else:
    print("\n3.  CRS já está unificado")

# 4. Criar código de município de 6 dígitos (IBGE) se não existir
if 'cod_mun_ibge' not in gdf_setores.columns:
    gdf_setores['cod_mun_ibge'] = gdf_setores['CD_MUN'].astype(str).str[:6]
    print(f"    Criada coluna 'cod_mun_ibge' a partir de CD_MUN")

# 5. Visualizar amostra
print("\n Amostra de médicos:")
display(gdf_medicos[['cnes', 'profissional_cns', 'municipio', 'geometry']].head())

print("\n Amostra de setores (com população):")
display(gdf_setores[['CD_SETOR', 'NM_MUN', 'v0001', 'AREA_KM2', 'cod_mun_ibge', 'geometry']].head())

In [ ]:
# ============================================================
# Spatial Join: Médicos DENTRO dos Setores (preciso!)
# ============================================================

print("=" * 70)
print("SPATIAL JOIN: MÉDICOS DENTRO DOS SETORES")
print("=" * 70)

# 1. Spatial join usando 'within' (médico dentro do polígono do setor)
print("\n1. Fazendo spatial join (médicos dentro dos setores)...")
print("Processando...")

gdf_medicos_setor = gpd.sjoin(
    gdf_medicos,
    gdf_setores[['CD_SETOR', 'NM_MUN', 'geometry']],
    how='left',
    predicate='within'
)

# Remover coluna de índice duplicada
if 'index_right' in gdf_medicos_setor.columns:
    gdf_medicos_setor = gdf_medicos_setor.drop(columns=['index_right'])

print(f"    Spatial join concluído: {len(gdf_medicos_setor):,} registros")

# 2. Verificar cobertura
total_medicos = len(gdf_medicos_setor)
medicos_com_setor = gdf_medicos_setor['CD_SETOR'].notna().sum()
medicos_sem_setor = total_medicos - medicos_com_setor

print(f"\n Cobertura do spatial join:")
print(f"    Médicos associados a setores: {medicos_com_setor:,} ({medicos_com_setor/total_medicos*100:.1f}%)")
print(f"    Médicos fora dos setores: {medicos_sem_setor:,} ({medicos_sem_setor/total_medicos*100:.1f}%)")

# 3. Salvar resultado intermediário
output_spatial = DATA_PROCESSED / "medicos_associados_setores_real.parquet"
gdf_medicos_setor.to_parquet(output_spatial, index=False)
print(f"\n Resultado salvo: {output_spatial.name}")

In [ ]:
# ============================================================
# Calcular ACESSIBILIDADE via E2SFCA (OTIMIZADO POR LOTES)
# ============================================================
# Método: Luo & Wang (2003) - Enhanced Two-Step Floating Catchment Area
# 
# Agora implementado em src/analysis/e2sfca.py (calcular_e2sfca).
# A chamada abaixo preserva os mesmos parâmetros e comportamento numérico.
#
# PASSO 1 (por CNES): R_j = médicos_j / Σ(pop_k × W(d_kj))
# PASSO 2 (por setor): A_i = Σ(R_j × W(d_ij))
#
# Resultado: índice contínuo de acessibilidade (médicos acessíveis por habitante)

print("=" * 70)
print("CÁLCULO DE ACESSIBILIDADE VIA E2SFCA (OTIMIZADO POR MUNICÍPIOS)")
print("=" * 70)

# versão original, mantida para conferência — ver src/analysis/e2sfca.py
# ---------------------------------------------------------------
# RAIO_CAPTURA_M = 5000  # 5 km
# BETA = 0.5             # Parâmetro de decaimento gaussiano
# CRS_PROJETADO = 'EPSG:31983'  # UTM Zone 23S (metros)
#
# def peso_gaussiano(distancia_m, raio_m=RAIO_CAPTURA_M, beta=BETA):
#     """Calcula peso gaussiano. Retorna 0 para distâncias >= raio."""
#     if distancia_m >= raio_m:
#         return 0.0
#     return np.exp(-beta * (distancia_m / raio_m) ** 2)
# ---------------------------------------------------------------

# Chamada da função extraída para src/analysis/e2sfca.py
acessibilidade_por_setor = calcular_e2sfca(
    gdf_medicos_setor,
    gdf_setores,
    raio_captura_m=5000,
    beta=0.5,
    crs_projetado='EPSG:31983',
)

# ============================================================
# Preparar intermediários para métricas legadas (Passo 5)
# ============================================================
# Estes GeoDataFrames são criados dentro de calcular_e2sfca() mas são
# necessários para as métricas legadas abaixo.

# Recriar aggregated CNES (mesmo procedimento do Passo 0)
cnes_agg = (
    gdf_medicos_setor
    .groupby(['cnes', 'municipio', 'nome_fantaia'])
    .agg(
        total_medicos_cnes=('profissional_cns', 'nunique'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first')
    )
    .reset_index()
)
gdf_cnes = gpd.GeoDataFrame(
    cnes_agg,
    geometry=gpd.points_from_xy(cnes_agg['longitude'], cnes_agg['latitude']),
    crs='EPSG:4326'
)
gdf_cnes_proj = gdf_cnes[gdf_cnes['total_medicos_cnes'] > 0].to_crs('EPSG:31983')

# Recriar centróides dos setores projetados
gdf_setores_proj = gdf_setores.to_crs('EPSG:31983')
gdf_setores_centroides = gdf_setores_proj.copy()
gdf_setores_centroides['geometry'] = gdf_setores_centroides.centroid

# ============================================================
# PASSO 4: Integrar com dados dos setores (mantendo colunas legadas)
# ============================================================
print("\n Integrando acessibilidade E2SFCA aos setores...")

# Merge com shapefile original
df_setores_com_e2sfca = gdf_setores.merge(
    acessibilidade_por_setor,
    on='CD_SETOR',
    how='left'
)

# Garantir que não há nulos
df_setores_com_e2sfca['acessibilidade_e2sfca'] = df_setores_com_e2sfca['acessibilidade_e2sfca'].fillna(0)
df_setores_com_e2sfca['categoria_acesso'] = df_setores_com_e2sfca['categoria_acesso'].fillna('6. Deserto médico (sem acesso)')

# ============================================================
# PASSO 5: Manter métricas legadas para compatibilidade
# ============================================================
print("\n Calculando métricas legadas (distância mínima, médicos no setor)...")

# Distância mínima ao médico mais próximo
nearest_result = gpd.sjoin_nearest(
    gdf_setores_centroides[['CD_SETOR', 'geometry']],
    gdf_cnes_proj[['cnes', 'geometry']],
    how='left',
    distance_col='dist_minima_metros'
)
dist_min = nearest_result.groupby('CD_SETOR')['dist_minima_metros'].min().reset_index()
df_setores_com_e2sfca = df_setores_com_e2sfca.merge(dist_min, on='CD_SETOR', how='left')
df_setores_com_e2sfca['dist_minima_metros'] = df_setores_com_e2sfca['dist_minima_metros'].fillna(99999)

# Médicos dentro do setor (spatial join original)
medicos_por_setor = (
    gdf_medicos_setor
    .groupby('CD_SETOR')
    .agg(
        total_medicos_dentro=('profissional_cns', 'nunique'),
        total_cnes_dentro=('cnes', 'nunique')
    )
    .reset_index()
)
df_setores_com_e2sfca = df_setores_com_e2sfca.merge(medicos_por_setor, on='CD_SETOR', how='left')
df_setores_com_e2sfca['total_medicos_dentro'] = df_setores_com_e2sfca['total_medicos_dentro'].fillna(0).astype(int)
df_setores_com_e2sfca['total_cnes_dentro'] = df_setores_com_e2sfca['total_cnes_dentro'].fillna(0).astype(int)

print(f"    {len(df_setores_com_e2sfca):,} setores com todas as métricas")

# SALVAR
output_acessibilidade = DATA_PROCESSED / "setores_com_acessibilidade_real.parquet"
df_setores_com_e2sfca.to_parquet(output_acessibilidade, index=False)
print(f"\n💾 Base de acessibilidade E2SFCA salva: {output_acessibilidade.name}")

print("\n" + "=" * 70)
print("RESUMO DO E2SFCA")
print("=" * 70)
print(f"   Raio de captura: 5 km")
print(f"   Função de decaimento: Gaussiana (β=0.5)")
print(f"   CNES analisados: {len(gdf_cnes_proj):,}")
print(f"   Setores analisados: {len(df_setores_com_e2sfca):,}")
print(f"   Índice médio E2SFCA: {df_setores_com_e2sfca['acessibilidade_e2sfca'].mean():.6f}")
print(f"   Setores em deserto: {(df_setores_com_e2sfca['categoria_acesso'] == '6. Deserto médico (sem acesso)').sum():,}")

In [ ]:
# ============================================================
#  VALIDADOR E2SFCA (100 SETORES + BORDA)
# ============================================================
# Agora implementado em src/validation/e2sfca.py.
# A chamada abaixo preserva os mesmos parâmetros e comportamento.

print("=" * 70)
print("VALIDAÇÃO ROBUSTA DO E2SFCA")
print("=" * 70)

# versão original, mantida para conferência — ver src/validation/e2sfca.py
# ---------------------------------------------------------------
# RAIO_M = 5000
# BETA = 0.5
# N_SETORES_TESTE = 100
# ATOL = 1e-9
# RTOL = 1e-4
#
# def sao_iguais(val1, val2, atol=ATOL, rtol=RTOL):
#     ...
# def erro_relativo(val1, val2):
#     ...
# def calcular_e2sfca_manual(cd_setor, gdf_setores_centroides, gdf_cnes_proj, raio=RAIO_M, beta=BETA):
#     ...
# ---------------------------------------------------------------

# Chamada da função extraída para src/validation/e2sfca.py
df_resultados = validar_e2sfca_100_setores(
    df_setores_com_e2sfca,
    gdf_setores,
    gdf_medicos_setor,
    n_setores=100,
    atol=1e-9,
    rtol=1e-4,
    seed=42,
)

In [ ]:
# ============================================================
# Identificar Desertos Médicos (com E2SFCA)
# ============================================================

print("=" * 70)
print("IDENTIFICAÇÃO DE DESERTOS MÉDICOS (E2SFCA)")
print("=" * 70)

# Detectar automaticamente a coluna de população
colunas = df_setores_com_e2sfca.columns.tolist()
COL_POP = next((c for c in colunas if c in ['v0001', 'POPULACAO', 'populacao']), None)
print(f" Coluna de população detectada: {COL_POP}")

# ============================================================
# 1. Desertos médicos (acessibilidade = 0)
# ============================================================
df_desertos = df_setores_com_e2sfca[
    df_setores_com_e2sfca['categoria_acesso'] == '6. Deserto médico (sem acesso)'
].copy()

print(f"\n️ Total de setores em deserto médico: {len(df_desertos):,}")
print(f"   População afetada: {df_desertos[COL_POP].sum():,}")
print(f"    % dos setores: {len(df_desertos)/len(df_setores_com_e2sfca)*100:.1f}%")
print(f"    % da população: {df_desertos[COL_POP].sum()/df_setores_com_e2sfca[COL_POP].sum()*100:.1f}%")

# Top 20 setores mais populosos em deserto médico
print(f"\n Top 20 setores mais populosos SEM acesso a médicos:")
top_desertos = df_desertos.nlargest(20, COL_POP)[
    ['CD_SETOR', 'NM_MUN', COL_POP, 'dist_minima_metros', 'categoria_acesso']
]
display(top_desertos)

# Análise por município
print(f"\n️ Municípios com mais setores em deserto médico:")
desertos_por_mun = df_desertos.groupby('NM_MUN').agg(
    num_setores_deserto=('CD_SETOR', 'count'),
    populacao_afetada=(COL_POP, 'sum'),
    dist_media=('dist_minima_metros', 'mean')
).reset_index()

print(f"\n   Top 10 municípios com piores indicadores:")
display(desertos_por_mun.nlargest(10, 'populacao_afetada'))

# ============================================================
# 2. Setores com acesso excelente
# ============================================================
df_excelente = df_setores_com_e2sfca[
    df_setores_com_e2sfca['categoria_acesso'] == '1. Excelente (acesso muito alto)'
].copy()

print(f"\n Setores com acesso excelente: {len(df_excelente):,}")
print(f"   População beneficiada: {df_excelente[COL_POP].sum():,}")

# ============================================================
# 3. Visualizações (atualizadas para E2SFCA)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Acessibilidade a Serviços de Saúde (E2SFCA)', fontsize=14, fontweight='bold')

# 3.1 Histograma do índice E2SFCA (NOVO - mostra a distribuição real)
ax1 = axes[0, 0]
idx_positivo = df_setores_com_e2sfca[df_setores_com_e2sfca['acessibilidade_e2sfca'] > 0]['acessibilidade_e2sfca']
ax1.hist(idx_positivo, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=idx_positivo.quantile(0.25), color='red', linestyle='--', label=f'P25 ({idx_positivo.quantile(0.25):.6f})')
ax1.axvline(x=idx_positivo.median(), color='orange', linestyle='--', label=f'Mediana ({idx_positivo.median():.6f})')
ax1.axvline(x=idx_positivo.quantile(0.75), color='green', linestyle='--', label=f'P75 ({idx_positivo.quantile(0.75):.6f})')
ax1.set_xlabel('Índice E2SFCA (médicos acessíveis por habitante)')
ax1.set_ylabel('Número de setores')
ax1.set_title('Distribuição do Índice E2SFCA (setores com acesso > 0)')
ax1.legend()

# 3.2 Pizza: Categorias de acesso E2SFCA
ax2 = axes[0, 1]
cores_categorias_e2sfca = {
    '1. Excelente (acesso muito alto)': '#1a9850',
    '2. Bom (acesso alto)': '#91cf60',
    '3. Moderado (acesso médio)': '#fee08b',
    '4. Limitado (acesso baixo)': '#fc8d59',
    '5. Crítico (acesso muito baixo)': '#d73027',
    '6. Deserto médico (sem acesso)': '#800000'
}

counts = df_setores_com_e2sfca['categoria_acesso'].value_counts()
mask = counts.index.isin(cores_categorias_e2sfca.keys())
counts_plot = counts[mask]
colors_plot = [cores_categorias_e2sfca[cat] for cat in counts_plot.index]

if not counts_plot.empty:
    ax2.pie(counts_plot, labels=counts_plot.index, colors=colors_plot, 
            autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribuição por categoria de acesso (E2SFCA)')

# 3.3 Histograma de distâncias 
ax3 = axes[1, 0]
dist_km = df_setores_com_e2sfca['dist_minima_metros'] / 1000
dist_km.hist(bins=50, ax=ax3, color='steelblue', edgecolor='black', alpha=0.7)
ax3.axvline(x=1, color='green', linestyle='--', label='1km')
ax3.axvline(x=5, color='orange', linestyle='--', label='5km')
ax3.axvline(x=10, color='red', linestyle='--', label='10km')
ax3.set_xlabel('Distância ao médico mais próximo (km)')
ax3.set_ylabel('Número de setores')
ax3.set_title('Distribuição de distâncias (métrica complementar)')
ax3.legend()

# 3.4 Scatter: Índice E2SFCA vs População do setor
ax4 = axes[1, 1]
df_scatter = df_setores_com_e2sfca[df_setores_com_e2sfca[COL_POP] > 0].copy()
ax4.scatter(
    df_scatter[COL_POP],
    df_scatter['acessibilidade_e2sfca'],
    alpha=0.3, s=10, color='steelblue', edgecolors='none'
)
ax4.set_xlabel('População do setor')
ax4.set_ylabel('Índice E2SFCA')
ax4.set_title('Relação População x Acessibilidade E2SFCA')
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.axhline(y=df_setores_com_e2sfca['acessibilidade_e2sfca'].median(), 
            color='red', linestyle='--', alpha=0.5, label='Mediana E2SFCA')
ax4.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 4. Salvar relatório
# ============================================================
output_relatorio = DATA_PROCESSED / "desertos_medicos_e2sfca.csv"
df_desertos.to_csv(output_relatorio, index=False, encoding='utf-8-sig')
print(f"\n Relatório de desertos médicos salvo: {output_relatorio.name}")

# ============================================================
# 5. Resumo estatístico do E2SFCA
# ============================================================
print("\n" + "=" * 70)
print("RESUMO ESTATÍSTICO DO E2SFCA")
print("=" * 70)
print(f"\n Índice E2SFCA (todos os setores):")
print(f"   Média: {df_setores_com_e2sfca['acessibilidade_e2sfca'].mean():.6f}")
print(f"   Mediana: {df_setores_com_e2sfca['acessibilidade_e2sfca'].median():.6f}")
print(f"   Desvio padrão: {df_setores_com_e2sfca['acessibilidade_e2sfca'].std():.6f}")
print(f"   Mínimo: {df_setores_com_e2sfca['acessibilidade_e2sfca'].min():.6f}")
print(f"   Máximo: {df_setores_com_e2sfca['acessibilidade_e2sfca'].max():.6f}")

print(f"\n Distribuição por categoria:")
for cat in sorted(df_setores_com_e2sfca['categoria_acesso'].unique()):
    count = len(df_setores_com_e2sfca[df_setores_com_e2sfca['categoria_acesso'] == cat])
    pct = count / len(df_setores_com_e2sfca) * 100
    pop = df_setores_com_e2sfca[df_setores_com_e2sfca['categoria_acesso'] == cat][COL_POP].sum()
    print(f"   {cat:50s}: {count:>8,} ({pct:5.1f}%) | Pop: {pop:>12,}")

In [ ]:
# ============================================================
# Comparar análise municipal vs acessibilidade setorial (E2SFCA)
# ============================================================
# Comparação: densidade municipal (médicos/1k hab) 
#             vs acessibilidade E2SFCA (setor)

print("=" * 70)
print("COMPARAÇÃO: DENSIDADE MUNICIPAL vs ACESSIBILIDADE E2SFCA")
print("=" * 70)

if df_municipal is None:
    print(" Base municipal não disponível para comparação")
else:
    # Verificar nomes das colunas
    print("\n Verificando colunas...")
    colunas_shapefile = df_setores_com_e2sfca.columns.tolist()
    print(f"   Colunas disponíveis: {colunas_shapefile[:10]}...")
    
    # Identificar coluna de população
    col_pop = next((c for c in ['v0001', 'POPULACAO', 'populacao'] if c in colunas_shapefile), None)
    col_mun = next((c for c in ['CD_MUN', 'cd_mun', 'cod_mun_ibge'] if c in colunas_shapefile), None)
    col_setor = next((c for c in ['CD_SETOR', 'cd_setor'] if c in colunas_shapefile), None)
    
    print(f"    Coluna de população: {col_pop}")
    print(f"    Coluna de município: {col_mun}")
    print(f"    Coluna de setor: {col_setor}")
    
    # Criar código de município de 6 dígitos se não existir
    if 'cod_mun_ibge' not in df_setores_com_e2sfca.columns:
        df_setores_com_e2sfca['cod_mun_ibge'] = df_setores_com_e2sfca[col_mun].astype(str).str[:6]
        print(f"    Criada coluna 'cod_mun_ibge' a partir de {col_mun}")
    
    # 1. Agregar dados de acessibilidade por município
    print("\n1. Agregando acessibilidade E2SFCA por município...")
    acessibilidade_por_mun = df_setores_com_e2sfca.groupby('cod_mun_ibge').agg(
        num_setores=(col_setor, 'count'),
        populacao_setorial=(col_pop, 'sum'),
        # Métricas legadas (distância)
        dist_media=('dist_minima_metros', 'mean'),
        dist_mediana=('dist_minima_metros', 'median'),
        dist_max=('dist_minima_metros', 'max'),
        # Métricas E2SFCA (NOVAS)
        acessibilidade_e2sfca_media=('acessibilidade_e2sfca', 'mean'),
        acessibilidade_e2sfca_mediana=('acessibilidade_e2sfca', 'median'),
        acessibilidade_e2sfca_max=('acessibilidade_e2sfca', 'max'),
        # Contagens por categoria E2SFCA
        num_setores_deserto=('categoria_acesso', lambda x: (x == '6. Deserto médico (sem acesso)').sum()),
        num_setores_critico=('categoria_acesso', lambda x: (x == '5. Crítico (acesso muito baixo)').sum()),
        num_setores_limitado=('categoria_acesso', lambda x: (x == '4. Limitado (acesso baixo)').sum()),
        num_setores_moderado=('categoria_acesso', lambda x: (x == '3. Moderado (acesso médio)').sum()),
        num_setores_bom=('categoria_acesso', lambda x: (x == '2. Bom (acesso alto)').sum()),
        num_setores_excelente=('categoria_acesso', lambda x: (x == '1. Excelente (acesso muito alto)').sum()),
        medicos_dentro_setor=('total_medicos_dentro', 'sum')
    ).reset_index()
    
    # Calcular percentuais
    acessibilidade_por_mun['pct_setores_deserto'] = (
        acessibilidade_por_mun['num_setores_deserto'] / acessibilidade_por_mun['num_setores'] * 100
    ).round(1)
    
    acessibilidade_por_mun['pct_setores_critico'] = (
        acessibilidade_por_mun['num_setores_critico'] / acessibilidade_por_mun['num_setores'] * 100
    ).round(1)
    
    acessibilidade_por_mun['pct_setores_excelente'] = (
        acessibilidade_por_mun['num_setores_excelente'] / acessibilidade_por_mun['num_setores'] * 100
    ).round(1)
    
    # Percentual de setores com acesso baixo (deserto + crítico + limitado)
    acessibilidade_por_mun['pct_acesso_baixo'] = (
        (acessibilidade_por_mun['num_setores_deserto'] + 
         acessibilidade_por_mun['num_setores_critico'] + 
         acessibilidade_por_mun['num_setores_limitado']) / 
        acessibilidade_por_mun['num_setores'] * 100
    ).round(1)
    
    # Converter distâncias para km
    acessibilidade_por_mun['dist_media_km'] = (acessibilidade_por_mun['dist_media'] / 1000).round(2)
    acessibilidade_por_mun['dist_mediana_km'] = (acessibilidade_por_mun['dist_mediana'] / 1000).round(2)
    acessibilidade_por_mun['dist_max_km'] = (acessibilidade_por_mun['dist_max'] / 1000).round(2)
    
    print(f"    {len(acessibilidade_por_mun):,} municípios agregados")
    
    # 2. Merge com base municipal
    print("\n2. Cruzando com base municipal...")
    df_comparacao = pd.merge(
        df_municipal[['cod_mun_ibge', 'nm_mun', 'total_medicos', 'populacao', 'medicos_por_1k', 'categoria_densidade']],
        acessibilidade_por_mun[['cod_mun_ibge', 'num_setores', 'populacao_setorial', 
                                 'dist_media_km', 'dist_mediana_km', 'dist_max_km',
                                 'acessibilidade_e2sfca_media', 'acessibilidade_e2sfca_mediana',
                                 'pct_setores_deserto', 'pct_setores_critico', 'pct_setores_excelente',
                                 'pct_acesso_baixo', 'medicos_dentro_setor']],
        on='cod_mun_ibge',
        how='inner'
    )
    
    print(f"\n Comparação concluída: {len(df_comparacao):,} municípios")
    
    # 3. Estatísticas gerais
    print(f"\n Estatísticas gerais:")
    print(f"   Densidade médica média (municipal): {df_comparacao['medicos_por_1k'].mean():.2f} médicos/1k hab")
    print(f"   Distância média ao médico (setorial): {df_comparacao['dist_media_km'].mean():.2f} km")
    print(f"   Distância mediana ao médico (setorial): {df_comparacao['dist_mediana_km'].median():.2f} km")
    print(f"   Índice E2SFCA médio: {df_comparacao['acessibilidade_e2sfca_media'].mean():.6f}")
    print(f"   Índice E2SFCA mediano: {df_comparacao['acessibilidade_e2sfca_mediana'].median():.6f}")
    print(f"   % médio de setores em deserto: {df_comparacao['pct_setores_deserto'].mean():.1f}%")
    print(f"   % médio de setores com acesso baixo: {df_comparacao['pct_acesso_baixo'].mean():.1f}%")
    print(f"   Média de médicos DENTRO dos setores: {df_comparacao['medicos_dentro_setor'].mean():.1f}")
    
    # 4. Correlações
    print(f"\n Correlações:")
    corr_densidade_dist = df_comparacao['medicos_por_1k'].corr(df_comparacao['dist_media_km'])
    print(f"   Densidade x Distância média: {corr_densidade_dist:.3f}")
    
    corr_densidade_e2sfca = df_comparacao['medicos_por_1k'].corr(df_comparacao['acessibilidade_e2sfca_media'])
    print(f"   Densidade x E2SFCA médio: {corr_densidade_e2sfca:.3f}")
    
    corr_densidade_deserto = df_comparacao['medicos_por_1k'].corr(df_comparacao['pct_setores_deserto'])
    print(f"   Densidade x % setores deserto: {corr_densidade_deserto:.3f}")
    
    corr_densidade_acesso_baixo = df_comparacao['medicos_por_1k'].corr(df_comparacao['pct_acesso_baixo'])
    print(f"   Densidade x % acesso baixo: {corr_densidade_acesso_baixo:.3f}")
    
    # 5. Identificar municípios "enganosos" (boa densidade mas acesso ruim)
    print(f"\n Municípios 'ENGANOSOS' (boa densidade mas acesso E2SFCA baixo):")
    print(f"   Critério: densidade ≥ 2.0 E % acesso baixo > 30%")
    
    enganosos = df_comparacao[
        (df_comparacao['medicos_por_1k'] >= 2.0) & 
        (df_comparacao['pct_acesso_baixo'] > 30.0)
    ].copy()
    
    if len(enganosos) > 0:
        print(f"\n   Encontrados: {len(enganosos):,} municípios")
        print(f"   População afetada: {enganosos['populacao'].sum():,}")
        print(f"\n   Top 10 casos:")
        display(
            enganosos.nlargest(10, 'pct_acesso_baixo')[
                ['nm_mun', 'medicos_por_1k', 'acessibilidade_e2sfca_mediana', 
                 'pct_acesso_baixo', 'pct_setores_deserto', 'populacao']
            ]
        )
    else:
        print(f"    Nenhum município enganoso encontrado")
    
    # 6. Municípios com boa acessibilidade
    print(f"\n Municípios com excelente acessibilidade:")
    print(f"   Critério: densidade ≥ 2.0 E % acesso baixo ≤ 10%")
    
    excelentes = df_comparacao[
        (df_comparacao['medicos_por_1k'] >= 2.0) & 
        (df_comparacao['pct_acesso_baixo'] <= 10.0)
    ].copy()
    
    if len(excelentes) > 0:
        print(f"   Encontrados: {len(excelentes):,} municípios")
        print(f"   População beneficiada: {excelentes['populacao'].sum():,}")
    
    # 7. Visualização da comparação
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Comparação: Densidade Municipal vs Acessibilidade E2SFCA', 
                 fontsize=14, fontweight='bold')
    
    # Scatter: Densidade vs E2SFCA
    ax1 = axes[0, 0]
    ax1.scatter(
        df_comparacao['medicos_por_1k'],
        df_comparacao['acessibilidade_e2sfca_mediana'],
        alpha=0.4, s=30, color='steelblue', edgecolors='none'
    )
    ax1.set_xlabel('Densidade médica (médicos/1k hab)')
    ax1.set_ylabel('Índice E2SFCA mediano')
    ax1.set_title('Relação Densidade x E2SFCA')
    ax1.set_xscale('log')
    ax1.axhline(y=df_comparacao['acessibilidade_e2sfca_mediana'].median(), 
                color='red', linestyle='--', alpha=0.5, label='Mediana E2SFCA')
    ax1.legend()
    
    # Scatter: Densidade vs % acesso baixo
    ax2 = axes[0, 1]
    ax2.scatter(
        df_comparacao['medicos_por_1k'],
        df_comparacao['pct_acesso_baixo'],
        alpha=0.4, s=30, color='steelblue', edgecolors='none'
    )
    ax2.set_xlabel('Densidade médica (médicos/1k hab)')
    ax2.set_ylabel('% setores com acesso baixo')
    ax2.set_title('Relação Densidade x % Acesso Baixo')
    ax2.set_xscale('log')
    ax2.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30% (limite enganoso)')
    ax2.legend()
    
    # Histograma: % de setores deserto
    ax3 = axes[1, 0]
    ax3.hist(
        df_comparacao['pct_setores_deserto'], 
        bins=30, color='steelblue', edgecolor='black', alpha=0.7
    )
    ax3.set_xlabel('% de setores em deserto médico')
    ax3.set_ylabel('Número de municípios')
    ax3.set_title('Distribuição de % setores deserto')
    ax3.axvline(x=10, color='red', linestyle='--', label='10%')
    ax3.legend()
    
    # Histograma: Índice E2SFCA mediano por município
    ax4 = axes[1, 1]
    ax4.hist(
        df_comparacao['acessibilidade_e2sfca_mediana'], 
        bins=30, color='steelblue', edgecolor='black', alpha=0.7
    )
    ax4.set_xlabel('Índice E2SFCA mediano por município')
    ax4.set_ylabel('Número de municípios')
    ax4.set_title('Distribuição do Índice E2SFCA Municipal')
    ax4.axvline(x=df_comparacao['acessibilidade_e2sfca_mediana'].median(), 
                color='red', linestyle='--', label='Mediana')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 8. Salvar comparação
    output_comparacao = DATA_PROCESSED / "comparacao_municipal_vs_e2sfca.parquet"
    df_comparacao.to_parquet(output_comparacao, index=False)
    print(f"\n Comparação salva: {output_comparacao.name}")

In [ ]:
# ============================================================
#  Visualizações da Análise de Acessibilidade (E2SFCA)
# ============================================================

# Usar o DataFrame com os dados do E2SFCA
df_plot_e2sfca = df_setores_com_e2sfca.copy()

# Verificar coluna de população
col_pop = next((c for c in ['v0001', 'POPULACAO', 'populacao'] if c in df_plot_e2sfca.columns), None)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Análise de Acessibilidade: Método E2SFCA (Oferta vs Demanda vs Distância)', 
             fontsize=14, fontweight='bold')

categorias_e2sfca = [
    '1. Excelente (acesso muito alto)',
    '2. Bom (acesso alto)',
    '3. Moderado (acesso médio)',
    '4. Limitado (acesso baixo)',
    '5. Crítico (acesso muito baixo)',
    '6. Deserto médico (sem acesso)'
]

cores_e2sfca = {
    '1. Excelente (acesso muito alto)': '#1a9850',
    '2. Bom (acesso alto)': '#91cf60',
    '3. Moderado (acesso médio)': '#fee08b',
    '4. Limitado (acesso baixo)': '#fc8d59',
    '5. Crítico (acesso muito baixo)': '#d73027',
    '6. Deserto médico (sem acesso)': '#800000'
}

# ============================================================
# 1. Histograma: Distribuição do Índice E2SFCA
# ============================================================
ax1 = axes[0, 0]
# Filtrar apenas valores > 0 para o histograma (desertos são 0 e distorcem a escala)
idx_positivo = df_plot_e2sfca[df_plot_e2sfca['acessibilidade_e2sfca'] > 0]['acessibilidade_e2sfca']

ax1.hist(idx_positivo, bins=50, color='steelblue', edgecolor='black', alpha=0.7)

# Adicionar linhas dos percentis usados na classificação
p25 = idx_positivo.quantile(0.25)
p50 = idx_positivo.quantile(0.50)
p75 = idx_positivo.quantile(0.75)

ax1.axvline(x=p25, color='red', linestyle='--', alpha=0.7, label=f'P25 ({p25:.5f})')
ax1.axvline(x=p50, color='orange', linestyle='--', alpha=0.7, label=f'Mediana ({p50:.5f})')
ax1.axvline(x=p75, color='green', linestyle='--', alpha=0.7, label=f'P75 ({p75:.5f})')

ax1.set_xlabel('Índice E2SFCA (médicos acessíveis por habitante)')
ax1.set_ylabel('Número de setores')
ax1.set_title('Distribuição do Índice E2SFCA (excluindo desertos = 0)')
ax1.legend()

# ============================================================
# 2. Boxplot: Índice E2SFCA por Categoria
# ============================================================
ax2 = axes[0, 1]

df_box = df_plot_e2sfca.copy()
df_box['categoria_acesso'] = pd.Categorical(
    df_box['categoria_acesso'], 
    categories=categorias_e2sfca, 
    ordered=True
)

# Boxplot do índice por categoria
df_box.boxplot(column='acessibilidade_e2sfca', by='categoria_acesso', ax=ax2, rot=45, patch_artist=True)

# Colorir as caixas do boxplot
for patch, category in zip(ax2.patches, categorias_e2sfca):
    patch.set_facecolor(cores_e2sfca[category])
    patch.set_alpha(0.7)

ax2.set_title('Distribuição do Índice E2SFCA por Categoria')
ax2.set_ylabel('Índice E2SFCA')
ax2.set_xlabel('') # Remove o rótulo automático do boxplot

# ============================================================
# 3. Scatter: População do Setor vs Índice E2SFCA
# ============================================================
ax3 = axes[1, 0]

df_scatter = df_plot_e2sfca[df_plot_e2sfca[col_pop] > 0].copy()

ax3.scatter(
    df_scatter[col_pop],
    df_scatter['acessibilidade_e2sfca'],
    alpha=0.3, s=10, color='steelblue', edgecolors='none'
)

ax3.set_xlabel('População do Setor')
ax3.set_ylabel('Índice E2SFCA')
ax3.set_title('Relação População x Acessibilidade E2SFCA')
ax3.set_xscale('log')
# Sem log no Y porque tem muitos zeros (desertos)

# Linha da mediana estadual
mediana_geral = df_plot_e2sfca['acessibilidade_e2sfca'].median()
ax3.axhline(y=mediana_geral, color='red', linestyle='--', alpha=0.7, label=f'Mediana ({mediana_geral:.5f})')
ax3.legend()

# ============================================================
# 4. Gráfico de Pizza: Distribuição por Categoria E2SFCA
# ============================================================
ax4 = axes[1, 1]

counts = df_plot_e2sfca['categoria_acesso'].value_counts()
mask = counts.index.isin(cores_e2sfca.keys())
counts_plot = counts[mask]
colors_plot = [cores_e2sfca[cat] for cat in counts_plot.index]

if not counts_plot.empty:
    wedges, texts, autotexts = ax4.pie(
        counts_plot, 
        labels=counts_plot.index, 
        colors=colors_plot, 
        autopct='%1.1f%%', 
        startangle=90,
        wedgeprops={'edgecolor': 'black', 'linewidth': 1}
    )
    # Melhorar legibilidade dos textos
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(9)
        autotext.set_color('black')

ax4.set_title('Distribuição de Setores por Categoria de Acesso (E2SFCA)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Preparar Dados para Visualização no Notebook 04
# ============================================================

print("=" * 70)
print("PREPARANDO DADOS PARA VISUALIZAÇÃO")
print("=" * 70)

# 1. Base de setores com acessibilidade E2SFCA (já pronta)
print("\n  Base de setores com acessibilidade E2SFCA:")
print(f"   Arquivo: setores_com_acessibilidade_real.parquet")
print(f"   Registros: {len(df_setores_com_e2sfca):,}")
print(f"   Colunas principais:")
print(f"      • CD_SETOR, NM_MUN, v0001 (população)")
print(f"      • acessibilidade_e2sfca (índice contínuo)")
print(f"      • categoria_acesso (6 categorias E2SFCA)")
print(f"      • dist_minima_metros (métrica complementar)")
print(f"      • total_medicos_dentro, total_cnes_dentro")

# 2. Base de médicos agregados por CNES (para mapa de pontos)
print("\n2. Agregando médicos por CNES...")
cnes_agg = (
    gdf_medicos_setor
    .groupby(['cnes', 'municipio', 'nome_fantaia'])
    .agg(
        total_medicos=('profissional_cns', 'nunique'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first')
    )
    .reset_index()
)

output_cnes_agg = DATA_PROCESSED / "cnes_agregados.parquet"
cnes_agg.to_parquet(output_cnes_agg, index=False)
print(f"   CNES agregados salvos: {len(cnes_agg):,} estabelecimentos")
print(f"   Arquivo: cnes_agregados.parquet")

# 3. Base municipal para mapa coroplético
if df_municipal is not None:
    output_municipal_mapa = DATA_PROCESSED / "base_municipal_para_mapa.parquet"
    df_municipal.to_parquet(output_municipal_mapa, index=False)
    print(f"\n Base municipal para mapa salva: {len(df_municipal):,} municípios")
    print(f"   Arquivo: base_municipal_para_mapa.parquet")

